In [2]:
from pathlib import Path
import math

import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit.Chem import Descriptors, Draw, PandasTools

In [3]:
#01 Menentukan direktori kerja saat ini
HERE = Path.cwd()
# Menyiapkan path ke folder data
DATA = HERE / "data"

In [ ]:
#02 Baca data dari analilis sebelumnya
molecules = pd.read_csv(DATA / "01sampledIndonesiaPlantNP.csv", index_col=0)
print(molecules.shape)
molecules.head()

In [8]:
#03 Membuat fungsi untuk kalkusi ro5
def calculate_ro5_properties(smiles):
    """
    Memeriksa apakah molekul (dalam format SMILES) memenuhi aturan Lipinski.

    Parameters
    ----------
    smiles : str
        SMILES untuk suatu molekul.

    Returns
    -------
    pandas.Series
        Berat molekul, jumlah akseptor/donor ikatan hidrogen, nilai logP,
        dan keterangan apakah molekul memenuhi aturan Lipinski.
    """
    # Membuat objek molekul dari SMILES menggunakan RDKit
    molecule = Chem.MolFromSmiles(smiles)
    # Menghitung properti kimia yang relevan dengan aturan Lipinski
    molecular_weight = Descriptors.ExactMolWt(molecule)
    n_hba = Descriptors.NumHAcceptors(molecule)
    n_hbd = Descriptors.NumHDonors(molecule)
    logp = Descriptors.MolLogP(molecule)
    # Memeriksa apakah molekul memenuhi aturan Lipinski
    conditions = [molecular_weight <= 500, n_hba <= 10, n_hbd <= 5, logp <= 5]
    ro5_fulfilled = sum(conditions) >= 3
    # Mengembalikan hasil perhitungan dalam bentuk pandas Series
    return pd.Series(
        [molecular_weight, n_hba, n_hbd, logp, ro5_fulfilled],
        index=["molecular_weight", "n_hba", "n_hbd", "logp", "ro5_fulfilled"],
    )

In [ ]:
#04 Menghitung properti Ro5 untuk setiap molekul berdasarkan kolom SMILES
ro5_properties = molecules["smiles"].apply(calculate_ro5_properties)
# Menampilkan beberapa baris pertama dari hasil perhitungan properti Ro5
ro5_properties.head()

In [ ]:
#05 Menggabungkan data molekul dengan properti Aturan Lipinski
molecules = pd.concat([molecules, ro5_properties], axis=1)
molecules.head()

In [12]:
#06 Memisahkan molekul yang memenuhi dan tidak memenuhi aturan Lipinski
molecules_ro5_fulfilled = molecules[molecules["ro5_fulfilled"]]
molecules_ro5_violated = molecules[~molecules["ro5_fulfilled"]]

# Menampilkan jumlah molekul sebelum dan sesudah difilter berdasarkan aturan Lipinski
print(f"# Molekul dalam data awal: {molecules.shape[0]}")
print(f"# Molekul yang memenuhi aturan Lipinski: {molecules_ro5_fulfilled.shape[0]}")
print(f"# Molekul yang tidak memenuhi aturan Lipinski: {molecules_ro5_violated.shape[0]}")

# Molekul dalam data awal: 200
# Molekul yang memenuhi aturan Lipinski: 155
# Molekul yang tidak memenuhi aturan Lipinski: 45


In [ ]:
#07 save output data dalam file
#Mengatur ulang indeks untuk dataframe hasil filtrasi
molecules_ro5_fulfilled.reset_index(drop=True, inplace=True)
# Menyimpan hasil filtrasi data molekul yang memenuhi aturan Lipinski ke file CSV
molecules_ro5_fulfilled.to_csv(DATA / "02NPlipinskiFulfilled.csv")
molecules_ro5_fulfilled.head()